In [1]:
import numpy as np
import itertools
from pyspark.sql import SparkSession
import time

In [2]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Ali'sSimHash") \
    .master("local[*]") \
    .config("spark.ui.showConsoleProgress", "false") \
    .getOrCreate()


In [3]:
def read_fvecs(path):
    data = np.fromfile(path, dtype=np.int32)
    d = data[0]
    data = data.reshape(-1, d+1)
    return data[:, 1:].view(np.float32)

def read_ivecs(path):
    data = np.fromfile(path, dtype=np.int32)
    d = data[0]
    data = data.reshape(-1, d+1)
    return data[:, 1:]

In [4]:
def random_normal_matrix(m, d, seed = 42):
    return np.random.default_rng(seed).standard_normal((m, d)).astype(np.float32)

def compute_signature(q, hyperPlanesMatrix):

    results = hyperPlanesMatrix @ q.T
    sig = (results > 0).T
    return  sig
def signature_to_band(signature_1d, b):

    m = signature_1d.shape[0]
    r = m // b
    bands = []
    for idx in range(b):
        start = idx * r
        end = (idx + 1) * r
        band_bits = signature_1d[start:end].astype(np.uint8)
        if r <= 64:
            bit_str = "".join(band_bits.astype(str))
            val = int(bit_str, 2)
        else:
            bit_str = "".join(band_bits.astype(str))
            val = int(bit_str, 2)
        bands.append((idx, val))
    return bands
def emit_bands(item, b):
        doc_id, signature = item
        signatures = np.array(signature, dtype=bool)
        # doc_ids ia a list of doc ids in the same band bucket
        res = [((bi, bv), doc_id) for bi, bv in signature_to_band(signatures, b)]
        return res

def emit_pairs(item):
     _, doc_ids = item
     return [(min(a, b_), max(a, b_)) for a, b_ in itertools.combinations(doc_ids, 2)]

In [5]:
def init_run_simhash(sparkSession, vectors, m, b, seed=42):
    num_dims = vectors.shape[1]
    hyperPlanes = random_normal_matrix(m, num_dims, seed)

    # compute signatures: (n_docs, m)
    signatures = compute_signature(vectors, hyperPlanes)
    print("signatures shape:", signatures.shape)  # (n_docs, m)

    # enumerate documents
    indexed = list(enumerate(signatures.tolist()))  # list of (doc_id, signature_list)

    rdd = sparkSession.sparkContext.parallelize(indexed, numSlices=6)

    band_rdd = rdd.flatMap(lambda x: emit_bands(x, b))

    # group by band key and keep buckets with at least 2 docs
    grouped = (band_rdd.groupByKey()
                        .mapValues(list)
                        .filter(lambda x: len(x[1]) >= 2))

    # for all pairs within each bucket
    candidate_pairs = grouped.flatMap(emit_pairs).distinct().collect()

    return set(candidate_pairs)

In [6]:
SIFT_paths = {
    "small": {
        "base": "./data/small/siftsmall_base.fvecs",
        "gt": "./data/small/siftsmall_groundtruth.ivecs",
        "queries": "./data/small/siftsmall_query.fvecs"
    }
}

N_VECTORS = 500
THRESHOLD = 0.9
SEED = 42


mVals = [32]
bVals = [4]

mode = "small"

base = read_fvecs(SIFT_paths[mode]["base"])
queries = read_fvecs(SIFT_paths[mode]["queries"])
groundTruts = read_ivecs(SIFT_paths[mode]["gt"])

base_normalized = base / np.linalg.norm(base, axis=1, keepdims=True)
queries_normalized = queries / np.linalg.norm(queries, axis=1, keepdims=True)

print(f"Loaded data successfully.\nShape: {base_normalized.shape}")

Loaded data successfully.
Shape: (10000, 128)


In [7]:
#spark = SparkSession.builder.appName("Ali'sSimHash").master("local[*]").config("spark.ui.showConsoleProgress", "false").getOrCreate()


for m in mVals:
    for b in bVals:
        if (m % b != 0):
            # illegal pair,
            continue
        
        r = m / b
        t0 = time.perf_counter()
        candidates = init_run_simhash(spark, queries, m, b, seed=SEED)
        t0 = time.perf_counter() - t0
        print(t0)


signatures shape: (100, 32)
25.627996199997142


In [8]:
candidates

{(0, 1),
 (0, 5),
 (0, 6),
 (0, 10),
 (0, 12),
 (0, 16),
 (0, 19),
 (0, 36),
 (0, 41),
 (0, 44),
 (0, 46),
 (0, 47),
 (0, 54),
 (0, 55),
 (0, 65),
 (0, 80),
 (0, 88),
 (0, 96),
 (1, 5),
 (1, 6),
 (1, 10),
 (1, 11),
 (1, 12),
 (1, 16),
 (1, 19),
 (1, 20),
 (1, 27),
 (1, 36),
 (1, 41),
 (1, 43),
 (1, 46),
 (1, 55),
 (1, 65),
 (1, 69),
 (1, 80),
 (1, 88),
 (2, 22),
 (2, 29),
 (2, 40),
 (2, 58),
 (2, 61),
 (2, 75),
 (2, 91),
 (2, 92),
 (2, 98),
 (3, 17),
 (3, 37),
 (3, 57),
 (3, 79),
 (3, 86),
 (3, 87),
 (3, 95),
 (4, 9),
 (4, 23),
 (4, 75),
 (5, 10),
 (5, 12),
 (5, 16),
 (5, 19),
 (5, 36),
 (5, 41),
 (5, 46),
 (5, 47),
 (5, 55),
 (5, 65),
 (5, 76),
 (5, 80),
 (5, 88),
 (6, 10),
 (6, 13),
 (6, 14),
 (6, 16),
 (6, 17),
 (6, 19),
 (6, 36),
 (6, 91),
 (6, 94),
 (7, 47),
 (7, 54),
 (7, 59),
 (7, 78),
 (7, 79),
 (7, 84),
 (7, 89),
 (8, 15),
 (8, 32),
 (8, 66),
 (8, 74),
 (8, 90),
 (9, 35),
 (9, 52),
 (9, 64),
 (9, 73),
 (9, 82),
 (9, 85),
 (10, 12),
 (10, 16),
 (10, 17),
 (10, 19),
 (10, 36),
 

In [9]:
groundTruts

array([[2176, 3752,  882, ...,  348, 3043, 3687],
       [2781, 9574, 2492, ..., 3849, 2905, 4102],
       [2707, 9938, 2698, ..., 1251, 8564, 8173],
       ...,
       [8825, 9081, 6142, ..., 8178, 5887, 4565],
       [5460, 5439, 5810, ..., 5199, 7483, 5232],
       [8082, 8782, 4767, ...,   11, 2482, 3631]],
      shape=(100, 100), dtype=int32)